In [8]:
#%pip install torch xgboost transformers textblob -q
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import yfinance as yf
from transformers import BertTokenizer, BertForSequenceClassification, BertModel
from textblob import TextBlob
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, accuracy_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"yfinance : {yf.__version__}")
print(f"PyTorch  : {torch.__version__}")


# ─────────────────────────────────────────────────────────────────────────────
# CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────
TICKERS     = ['0700.HK', '0005.HK', '1299.HK', '3690.HK']
START_DATE  = "2022-01-01"
END_DATE    = "2025-12-31"
WINDOW_SIZE = 10
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BERT_HIDDEN_DIM     = 768   # FinBERT CLS token embedding
POLYVALENT_EXTRA    = 3     # polarity + subjectivity + intensity
SENTIMENT_DIM       = BERT_HIDDEN_DIM + POLYVALENT_EXTRA   # = 771
PRICE_FEATURE_DIM   = 9     # OHLCV + RSI + MACD + BB_upper + BB_lower
TOTAL_FEATURE_DIM   = PRICE_FEATURE_DIM + SENTIMENT_DIM    # = 780 (extensible)

print(f"\nFeature dimensions:")
print(f"  BERT CLS embedding  : {BERT_HIDDEN_DIM}")
print(f"  Polyvalent extras   : {POLYVALENT_EXTRA}  (polarity + subjectivity + intensity)")
print(f"  Sentiment total     : {SENTIMENT_DIM}  ← 771-dim vector")
print(f"  Price features      : {PRICE_FEATURE_DIM}")
print(f"  Combined total      : {TOTAL_FEATURE_DIM}")


# ─────────────────────────────────────────────────────────────────────────────
# UTILITY: Clean yfinance download
# ─────────────────────────────────────────────────────────────────────────────
def download_clean(ticker: str, start: str, end: str) -> pd.DataFrame:
    raw = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    raw = raw.loc[:, ~raw.columns.duplicated()]
    raw = raw[pd.to_datetime(raw.index, errors="coerce").notna()]
    raw.index = pd.to_datetime(raw.index)
    raw.index.name = "Date"
    raw = raw.apply(pd.to_numeric, errors="coerce").dropna(how="all")
    required = {"Open", "High", "Low", "Close", "Volume"}
    missing = required - set(raw.columns)
    if missing:
        raise ValueError(f"[{ticker}] Missing columns: {missing}")
    return raw


# ─────────────────────────────────────────────────────────────────────────────
# TECHNICAL INDICATORS  (9 price features)
# ─────────────────────────────────────────────────────────────────────────────
def get_technical_indicators(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # RSI
    delta       = df['Close'].diff()
    gain        = delta.where(delta > 0, 0).rolling(14).mean()
    loss        = (-delta.where(delta < 0, 0)).rolling(14).mean()
    df['RSI']   = 100 - (100 / (1 + gain / loss))
    # MACD
    df['EMA12'] = df['Close'].ewm(span=12, adjust=False).mean()
    df['EMA26'] = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD']  = df['EMA12'] - df['EMA26']
    # Bollinger Bands
    ma              = df['Close'].rolling(20).mean()
    std             = df['Close'].rolling(20).std()
    df['BB_upper']  = ma + 2 * std
    df['BB_lower']  = ma - 2 * std
    # Drop intermediate EMA columns
    df.drop(columns=['EMA12', 'EMA26'], inplace=True)
    return df.bfill().ffill().dropna()


# ─────────────────────────────────────────────────────────────────────────────
# 771-DIMENSIONAL POLYVALENT SENTIMENT EXTRACTOR
# ─────────────────────────────────────────────────────────────────────────────
class PolyvalentExtractor771:
    """
    Produces a 771-dimensional sentiment vector per headline:

        [0:768]  — FinBERT CLS token hidden-state embedding
                   Captures deep contextual financial language semantics.
                   Dimension source: BertModel last_hidden_state[:, 0, :]

        [768]    — Polarity score (-1 negative, 0 neutral, +1 positive)
                   Derived from FinBERT classification head argmax.

        [769]    — Subjectivity score [0.0, 1.0]
                   Derived from TextBlob lexicon-based analysis.

        [770]    — Intensity score [0.0, 1.0]
                   Max softmax probability from FinBERT classification head.

    References: Malo et al. (2014); Loughran & McDonald (2011);
                Araci (2019) FinBERT; Tetlock (2007).
    """
    FINBERT_MODEL = 'ProsusAI/finbert'

    def __init__(self):
        print(f"Loading FinBERT tokenizer and models from '{self.FINBERT_MODEL}'...")

        self.tokenizer = BertTokenizer.from_pretrained(self.FINBERT_MODEL)

        # Classification head — for polarity + intensity (dims 768–770)
        self.clf_model = BertForSequenceClassification.from_pretrained(
            self.FINBERT_MODEL
        ).to(DEVICE).eval()

        # Base BERT encoder — for 768-dim CLS embedding (dims 0–767)
        self.emb_model = BertModel.from_pretrained(
            self.FINBERT_MODEL
        ).to(DEVICE).eval()

        print(f"PolyvalentExtractor771 ready on {DEVICE}. Output dim = {SENTIMENT_DIM}")

    def _tokenize(self, text: str) -> dict:
        return self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128
        ).to(DEVICE)

    def extract(self, text: str) -> np.ndarray:
        """
        Returns a numpy array of shape (771,).

        Parameters
        ----------
        text : str
            Raw headline or news sentence.

        Returns
        -------
        np.ndarray — shape (771,)
            Concatenation of [CLS_embedding(768) | polarity(1) |
                              subjectivity(1) | intensity(1)]
        """
        inputs = self._tokenize(text)

        with torch.no_grad():
            # ── Dims 0–767: CLS token hidden state ───────────────────
            emb_output = self.emb_model(**inputs)
            cls_embedding = emb_output.last_hidden_state[:, 0, :]   # (1, 768)
            cls_vec = cls_embedding.squeeze(0).cpu().numpy()         # (768,)

            # ── Classification head for polarity + intensity ──────────
            clf_output = self.clf_model(**inputs)
            probs = torch.nn.functional.softmax(
                clf_output.logits, dim=-1
            ).squeeze(0)  # (3,) → [negative, neutral, positive]

        # ── Dim 768: Polarity (-1, 0, +1) ────────────────────────────
        polarity = float(torch.argmax(probs).item()) - 1.0

        # ── Dim 769: Subjectivity [0, 1] ─────────────────────────────
        subjectivity = float(TextBlob(text).sentiment.subjectivity)

        # ── Dim 770: Intensity [0, 1] ─────────────────────────────────
        intensity = float(torch.max(probs).item())

        # ── Concatenate → (771,) ──────────────────────────────────────
        sentiment_vector = np.concatenate([
            cls_vec,                             # 768 dims
            np.array([polarity,                  # dim 768
                      subjectivity,              # dim 769
                      intensity])                # dim 770
        ])

        assert sentiment_vector.shape == (SENTIMENT_DIM,), (
            f"Expected ({SENTIMENT_DIM},), got {sentiment_vector.shape}"
        )
        return sentiment_vector

    def extract_batch(self, texts: list[str]) -> np.ndarray:
        """
        Batch extraction for a list of headlines.

        Returns
        -------
        np.ndarray — shape (n_texts, 771)
        """
        return np.vstack([self.extract(t) for t in texts])

    def aggregate_daily(self, texts: list[str]) -> np.ndarray:
        """
        Aggregate multiple headlines for one trading day into a single
        771-dim vector by mean-pooling across all headline vectors.

        Parameters
        ----------
        texts : list[str]
            All headlines published on a given trading day.

        Returns
        -------
        np.ndarray — shape (771,)
        """
        if not texts:
            return np.zeros(SENTIMENT_DIM, dtype=np.float32)
        vectors = self.extract_batch(texts)
        return vectors.mean(axis=0)              # mean-pool → (771,)


# ─────────────────────────────────────────────────────────────────────────────
# SENTIMENT ALIGNMENT: merge daily 771-dim vectors with price DataFrame
# ─────────────────────────────────────────────────────────────────────────────
def align_sentiment_to_prices(
    price_df: pd.DataFrame,
    sentiment_df: pd.DataFrame
) -> pd.DataFrame:
    """
    Left-join 771-dim daily sentiment vectors onto price+indicator DataFrame.
    Missing sentiment days (weekends/no-news) are forward-filled then zero-filled.

    Parameters
    ----------
    price_df     : pd.DataFrame  — index=Date, cols = OHLCV + RSI + MACD + BB
    sentiment_df : pd.DataFrame  — index=Date, cols = sent_0 … sent_770

    Returns
    -------
    pd.DataFrame — shape (n_days, 9 + 771) = (n_days, 780)
    """
    combined = price_df.join(sentiment_df, how="left")
    sent_cols = [c for c in combined.columns if c.startswith("sent_")]
    combined[sent_cols] = combined[sent_cols].ffill().fillna(0.0)
    return combined


def build_sentiment_df(
    headlines_by_date: dict,          # {"2022-01-03": ["headline1", ...], ...}
    extractor: PolyvalentExtractor771
) -> pd.DataFrame:
    """
    Convert a dict of {date_str: [headlines]} into a DataFrame of
    daily 771-dim sentiment vectors.

    Returns
    -------
    pd.DataFrame — index=DatetimeIndex, columns=sent_0…sent_770
    """
    records = {}
    for date_str, texts in headlines_by_date.items():
        records[date_str] = extractor.aggregate_daily(texts)

    sent_df = pd.DataFrame.from_dict(
        records, orient="index",
        columns=[f"sent_{i}" for i in range(SENTIMENT_DIM)]
    )
    sent_df.index = pd.to_datetime(sent_df.index)
    sent_df.index.name = "Date"
    return sent_df


# ─────────────────────────────────────────────────────────────────────────────
# SEQUENCE BUILDER: sliding window → (X, y) tensors
# ─────────────────────────────────────────────────────────────────────────────
def build_sequences(
    scaled: np.ndarray,
    window: int = WINDOW_SIZE
) -> tuple[np.ndarray, np.ndarray]:
    """
    Converts scaled feature matrix into overlapping sequences.

    Returns
    -------
    X : np.ndarray — shape (n_samples, window, n_features)
    y : np.ndarray — shape (n_samples,)  — next-day Close (scaled col 3)
    """
    X, y = [], []
    close_col_idx = 3               # 'Close' is the 4th column (0-indexed)
    for i in range(window, len(scaled)):
        X.append(scaled[i - window:i, :])
        y.append(scaled[i, close_col_idx])
    return np.array(X), np.array(y)


# ─────────────────────────────────────────────────────────────────────────────
# MODEL ARCHITECTURES
# ─────────────────────────────────────────────────────────────────────────────
class StockLSTM(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 128, num_layers: int = 2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim, hidden_dim, num_layers,
            batch_first=True, dropout=0.2
        )
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


class StockTransformer(nn.Module):
    def __init__(
        self,
        input_dim: int,
        nhead: int = 4,
        num_layers: int = 2,
        dim_feedforward: int = 256
    ):
        super().__init__()
        self.embedding = nn.Linear(input_dim, dim_feedforward)
        encoder_layer  = nn.TransformerEncoderLayer(
            d_model=dim_feedforward,
            nhead=nhead,
            dropout=0.1,
            batch_first=True               # avoids manual transpose
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(dim_feedforward, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x   = self.embedding(x)            # (batch, seq, dim_feedforward)
        out = self.transformer(x)
        return self.fc(out[:, -1, :])      # last timestep


# ─────────────────────────────────────────────────────────────────────────────
# PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
def run_forecasting_pipeline(
    ticker: str,
    headlines_by_date: dict = None,
    extractor: PolyvalentExtractor771 = None
):
    """
    Full pipeline:
      1. Download + clean price data
      2. Compute technical indicators   →  9-dim price features
      3. Align 771-dim sentiment vectors (or zeros if not provided)
      4. Scale → (n_days, 780) feature matrix
      5. Build sliding-window sequences → (X, y)
      6. Train/test split (80/20)

    Parameters
    ----------
    ticker            : HKEX ticker string e.g. '0700.HK'
    headlines_by_date : dict {date_str: [headlines]} — optional
    extractor         : PolyvalentExtractor771 instance — optional
    """
    print(f"\n{'─'*60}")
    print(f"  Pipeline: {ticker}")
    print(f"{'─'*60}")

    # ── 1. Price data ─────────────────────────────────────────────────
    price_df = download_clean(ticker, START_DATE, END_DATE)
    price_df = get_technical_indicators(price_df)
    print(f"  Price rows   : {len(price_df)} | Price features: {price_df.shape[1]}")

    # ── 2. Sentiment vectors (771-dim) ────────────────────────────────
    if extractor is not None and headlines_by_date is not None:
        sent_df  = build_sentiment_df(headlines_by_date, extractor)
        full_df  = align_sentiment_to_prices(price_df, sent_df)
        print(f"  Sentiment    : LIVE FinBERT 771-dim vectors aligned")
    else:
        # Pad with zeros when no live headlines are available
        # (synthetic corpus or ablation run)
        zero_sent = pd.DataFrame(
            np.zeros((len(price_df), SENTIMENT_DIM), dtype=np.float32),
            index=price_df.index,
            columns=[f"sent_{i}" for i in range(SENTIMENT_DIM)]
        )
        full_df  = pd.concat([price_df, zero_sent], axis=1)
        print(f"  Sentiment    : ZEROED (no headlines provided — ablation mode)")

    print(f"  Combined dim : {full_df.shape[1]}  "
          f"({PRICE_FEATURE_DIM} price + {SENTIMENT_DIM} sentiment = "
          f"{PRICE_FEATURE_DIM + SENTIMENT_DIM})")

    # ── 3. Scale ──────────────────────────────────────────────────────
    scaler      = MinMaxScaler()
    scaled      = scaler.fit_transform(full_df)              # (n_days, 780)

    # ── 4. Sliding-window sequences ───────────────────────────────────
    X, y        = build_sequences(scaled, window=WINDOW_SIZE)
    print(f"  Sequences    : X{X.shape}  y{y.shape}")

    # ── 5. Train / test split (80 % train, 20 % test) ─────────────────
    split       = int(len(X) * 0.8)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]
    print(f"  Train        : {len(X_train)} sequences")
    print(f"  Test         : {len(X_test)} sequences")

    # ── 6. Convert to tensors ─────────────────────────────────────────
    X_train_t   = torch.tensor(X_train, dtype=torch.float32).to(DEVICE)
    X_test_t    = torch.tensor(X_test,  dtype=torch.float32).to(DEVICE)
    y_train_t   = torch.tensor(y_train, dtype=torch.float32).to(DEVICE)
    y_test_t    = torch.tensor(y_test,  dtype=torch.float32).to(DEVICE)

    # ── 7. Instantiate models with correct input_dim ──────────────────
    input_dim   = X_train_t.shape[2]                         # 780
    lstm_model  = StockLSTM(input_dim=input_dim).to(DEVICE)
    trans_model = StockTransformer(input_dim=input_dim).to(DEVICE)

    print(f"\n  ✓ Models ready — input_dim = {input_dim}")
    print(f"    LSTM params       : {sum(p.numel() for p in lstm_model.parameters()):,}")
    print(f"    Transformer params: {sum(p.numel() for p in trans_model.parameters()):,}")

    return {
        "ticker":      ticker,
        "X_train":     X_train_t,
        "X_test":      X_test_t,
        "y_train":     y_train_t,
        "y_test":      y_test_t,
        "scaler":      scaler,
        "lstm":        lstm_model,
        "transformer": trans_model,
        "feature_cols": list(full_df.columns),
        "input_dim":   input_dim,
    }


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # ── Optionally initialise extractor once and reuse across tickers ──
    # extractor = PolyvalentExtractor771()

    results = {}
    for ticker in TICKERS:
        results[ticker] = run_forecasting_pipeline(
            ticker,
            headlines_by_date=None,    # replace with real dict from Section 4
            extractor=None             # replace with extractor instance
        )

yfinance : 0.2.63
PyTorch  : 2.11.0

Feature dimensions:
  BERT CLS embedding  : 768
  Polyvalent extras   : 3  (polarity + subjectivity + intensity)
  Sentiment total     : 771  ← 771-dim vector
  Price features      : 9
  Combined total      : 780

────────────────────────────────────────────────────────────
  Pipeline: 0700.HK
────────────────────────────────────────────────────────────
  Price rows   : 980 | Price features: 9
  Sentiment    : ZEROED (no headlines provided — ablation mode)
  Combined dim : 780  (9 price + 771 sentiment = 780)
  Sequences    : X(970, 10, 780)  y(970,)
  Train        : 776 sequences
  Test         : 194 sequences

  ✓ Models ready — input_dim = 780
    LSTM params       : 598,145
    Transformer params: 2,830,337

────────────────────────────────────────────────────────────
  Pipeline: 0005.HK
────────────────────────────────────────────────────────────
  Price rows   : 980 | Price features: 9
  Sentiment    : ZEROED (no headlines provided — ablation 